In [1]:
import requests
import json
import uuid
from IPython.display import display_javascript, display_html, display
import pandas as pd

response = requests.get("http://api.openweathermap.org/data/2.5/forecast?id=5780993&APPID=e0c55e00bf021f6142de334823046e9e&units=imperial")
weather = response.content.decode("utf-8")
weatherDict = json.loads(weather)

In [2]:
print(json.dumps(weatherDict, indent=2))

{
  "list": [
    {
      "main": {
        "humidity": 100,
        "pressure": 866.5,
        "grnd_level": 866.5,
        "temp_max": 36.45,
        "temp": 36.45,
        "sea_level": 1031.8,
        "temp_kf": 0.51,
        "temp_min": 35.53
      },
      "sys": {
        "pod": "d"
      },
      "dt_txt": "2016-11-17 21:00:00",
      "dt": 1479416400,
      "snow": {
        "3h": 0.234
      },
      "clouds": {
        "all": 88
      },
      "rain": {},
      "wind": {
        "deg": 317.508,
        "speed": 14.7
      },
      "weather": [
        {
          "main": "Snow",
          "id": 600,
          "icon": "13d",
          "description": "light snow"
        }
      ]
    },
    {
      "main": {
        "humidity": 100,
        "pressure": 868.75,
        "grnd_level": 868.75,
        "temp_max": 36.21,
        "temp": 36.21,
        "sea_level": 1035.43,
        "temp_kf": 0.38,
        "temp_min": 35.52
      },
      "sys": {
        "pod": "n"
      },
      "

In [3]:
# Method for rendering collapsible JSON from: http://stackoverflow.com/questions/18873066/pretty-json-formatting-in-ipython-notebook

class RenderJSON(object):
    def __init__(self, json_data):
        if isinstance(json_data, dict):
            self.json_str = json.dumps(json_data)
        else:
            self.json_str = json
        self.uuid = str(uuid.uuid4())

    def _ipython_display_(self):
        display_html('<div id="{}" style="height: 600px; width:100%;"></div>'.format(self.uuid),
        raw=True)
        
        display_javascript("""
        require(["https://rawgit.com/caldwell/renderjson/master/renderjson.js"], function() {
        document.getElementById('%s').appendChild(renderjson(%s))
        });
        """ % (self.uuid, self.json_str), raw=True)
        
RenderJSON(weatherDict)

In [4]:
#pdWeather = pd.read_json(weatherDict)
#pdWeather

#pd.DataFrame(weatherDict["data"], columns=[x["label"] for x in weatherDict["fields"]])

timeList = []
maxTempList = []
minTempList = []
pressureList = []
humidityList = []
tempList = []
wind_speedList = []
windDegList = []
rainList = []
rainListMM = []
snowList = []
snowListMM = []

for weatherEntry in weatherDict["list"]:
    timeList.append(weatherEntry["dt_txt"])
    mainWeather = weatherEntry["main"]
    maxTempList.append(mainWeather["temp_max"])
    minTempList.append(mainWeather["temp_min"])
    pressureList.append(mainWeather["pressure"])
    humidityList.append(mainWeather["humidity"])
    tempList.append(mainWeather["temp"])
    windWeather = weatherEntry["wind"]
    wind_speedList.append(windWeather["speed"])
    windDegList.append(windWeather["deg"])
    if "3h" in weatherEntry["rain"]:
        rainList.append(1)
        rainListMM.append(weatherEntry["rain"]["3h"])
    else:
        rainList.append(0)
        rainListMM.append(0)
    if "3h" in weatherEntry["snow"]:
        snowList.append(1)
        snowListMM.append(weatherEntry["snow"]["3h"])
    else:
        snowList.append(0)
        snowListMM.append(0)
        
    

In [5]:
data = [('DateTime', timeList),
         ('MaxTemp', maxTempList),
         ('MinTemp', minTempList),
         ('Pressure', pressureList),
         ('Humidity', humidityList),
         ('Temperature', tempList),
         ('WindSpeed', wind_speedList), 
         ('WindDeg', windDegList),
         ('Rain', rainList),
         ('Rain (mm)', rainListMM),
         ('Snow', snowList),
         ('Snow (mm)', snowListMM)
         ]

weatherForecast = pd.DataFrame.from_items(data)
weatherForecast["DateTime"] = weatherForecast["DateTime"].apply(lambda x: str(x)[:10])

finalForecast = weatherForecast.groupby("DateTime").mean()

finalForecast["MinTemp"] = weatherForecast.groupby("DateTime").min()["MinTemp"]
finalForecast["MaxTemp"] = weatherForecast.groupby("DateTime").max()["MaxTemp"]
finalForecast["MaxPressure"] = weatherForecast.groupby("DateTime").max()["Pressure"]
finalForecast["MinPressure"] = weatherForecast.groupby("DateTime").min()["Pressure"]
finalForecast["MaxWindSpeed"] = weatherForecast.groupby("DateTime").max()["WindSpeed"]
finalForecast["MinWindSpeed"] = weatherForecast.groupby("DateTime").min()["WindSpeed"]

finalForecast

,MaxTemp,MinTemp,Pressure,Humidity,Temperature,WindSpeed,WindDeg,Rain,Rain (mm),Snow,Snow (mm),MaxPressure,MinPressure,MaxWindSpeed,MinWindSpeed
DateTime,,,,,,,,,,,,,,,
2016-11-17,36.45,35.53,866.50000,100.000,36.450000,14.700000,317.508000,0.000,0.000000,1.000,0.234,866.50,866.50,14.70,14.70
2016-11-18,36.96,25.04,873.50875,100.000,31.917500,6.522500,176.815737,0.125,0.000937,0.375,0.020,876.55,868.75,10.00,1.32
2016-11-19,46.33,28.53,871.49875,100.000,33.695000,5.033750,158.626750,0.000,0.000000,0.000,0.000,873.51,868.73,7.76,2.71
2016-11-20,51.71,37.17,867.24250,98.375,41.538750,5.893750,165.125875,0.000,0.000000,0.000,0.000,867.99,865.68,9.04,3.18
2016-11-21,47.29,42.00,863.51875,97.625,45.122500,7.982500,178.938963,0.625,0.239062,0.000,0.000,864.83,862.33,13.91,2.17
2016-11-22,41.82,39.12,868.72000,100.000,40.125714,5.505714,287.215286,1.000,0.464286,0.000,0.000,873.80,863.60,7.83,2.89


In [6]:
finalForecast.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 2016-11-17 to 2016-11-22
Data columns (total 15 columns):
MaxTemp         6 non-null float64
MinTemp         6 non-null float64
Pressure        6 non-null float64
Humidity        6 non-null float64
Temperature     6 non-null float64
WindSpeed       6 non-null float64
WindDeg         6 non-null float64
Rain            6 non-null float64
Rain (mm)       6 non-null float64
Snow            6 non-null float64
Snow (mm)       6 non-null float64
MaxPressure     6 non-null float64
MinPressure     6 non-null float64
MaxWindSpeed    6 non-null float64
MinWindSpeed    6 non-null float64
dtypes: float64(15)
memory usage: 768.0+ bytes


In [7]:
finalForecast.describe()

,MaxTemp,MinTemp,Pressure,Humidity,Temperature,WindSpeed,WindDeg,Rain,Rain (mm),Snow,Snow (mm),MaxPressure,MinPressure,MaxWindSpeed,MinWindSpeed
count,6.000000,6.00000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,43.426667,34.56500,868.498125,99.333333,38.141577,7.606369,214.038435,0.291667,0.117381,0.229167,0.042333,870.530000,865.931667,10.540000,4.495000
std,6.083442,6.49448,3.592287,1.059678,5.011656,3.622155,69.485535,0.423281,0.194958,0.406330,0.094237,4.711607,2.629399,3.042256,5.042217
min,36.450000,25.04000,863.518750,97.625000,31.917500,5.033750,158.626750,0.000000,0.000000,0.000000,0.000000,864.830000,862.330000,7.760000,1.320000
25%,38.175000,30.28000,866.685625,98.781250,34.383750,5.602723,168.048341,0.000000,0.000000,0.000000,0.000000,866.872500,864.120000,8.132500,2.305000
50%,44.075000,36.35000,867.981250,100.000000,38.287857,6.208125,177.877350,0.062500,0.000469,0.000000,0.000000,870.750000,866.090000,9.520000,2.800000
75%,47.050000,38.63250,870.804063,100.000000,41.185491,7.617500,260.146205,0.500000,0.179531,0.281250,0.015000,873.727500,868.172500,12.932500,3.107500
max,51.710000,42.00000,873.508750,100.000000,45.122500,14.700000,317.508000,1.000000,0.464286,1.000000,0.234000,876.550000,868.750000,14.700000,14.700000
